# Chapter 14 — The Memory Contamination Problem

**Book alignment:** Hallucination From First Principles, Chapter 14

**Question this notebook isolates:** Do admission and read-path gates block self-citation circularity and stale reuse while permitting eligible evidence-indexed recall?

Synthetic fixtures with a perfect structured oracle demonstrate control invariants, not real verifier accuracy.


In [ ]:
from pathlib import Path
import sys
from dataclasses import dataclass, field
from typing import Optional

import numpy as np


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "hallucination-from-first-principles").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/hallucination-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "hallucination-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import recovery_demo as rec
import policy_engine as pe

rng = np.random.default_rng(14)

FACTUAL = "FACTUAL_EVIDENCE"


## Admission quarantines unverified model output

A naive writer stores every useful-looking statement as `ACTIVE`. The governed gate instead grants capabilities: `MODEL_OUTPUT + UNVERIFIED` becomes `QUARANTINED` (regression-test use only), while `EXTERNAL_SOURCE + VERIFIED` becomes `ACTIVE` with factual rights. A summary derived from one verified and one unverified parent stays quarantined: transformation does not upgrade provenance.


In [ ]:
@dataclass
class MemoryRecord:
    mid: str
    content: str
    origin: str
    verification: str
    lifecycle: str
    allowed_uses: list
    parents: list = field(default_factory=list)
    evidence_id: Optional[str] = None
    source_family: Optional[str] = None
    valid_to: Optional[str] = None


def admit(mid, content, origin, verification, parents=(), evidence_id=None,
          source_family=None, valid_to=None):
    ok_parents = list(parents)
    if origin == "EXTERNAL_SOURCE" and verification == "VERIFIED" and not ok_parents:
        return MemoryRecord(mid, content, origin, verification, "ACTIVE",
                              ["CONVERSATIONAL_CONTEXT", FACTUAL], [],
                              evidence_id, source_family, valid_to)
    if (verification == "VERIFIED" and ok_parents
            and all(FACTUAL in p.allowed_uses and p.lifecycle == "ACTIVE"
                    for p in ok_parents)):
        return MemoryRecord(mid, content, origin, verification, "ACTIVE",
                              ["CONVERSATIONAL_CONTEXT", FACTUAL], ok_parents,
                              evidence_id, source_family, valid_to)
    return MemoryRecord(mid, content, origin, verification, "QUARANTINED",
                          ["REGRESSION_TEST"], ok_parents,
                          evidence_id, source_family, valid_to)


def factual_recall(store, as_of="2026-08-30"):
    out = []
    for m in store.values():
        if m.lifecycle != "ACTIVE":
            continue
        if FACTUAL not in m.allowed_uses:
            continue
        if m.verification != "VERIFIED":
            continue
        if m.valid_to is not None and as_of > m.valid_to:
            continue
        out.append(m.mid)
    return out


flawed_store = {
    "m1": MemoryRecord("m1", "Q3 revenue was approximately $46 million.",
                           "MODEL_OUTPUT", "UNVERIFIED", "ACTIVE",
                           ["CONVERSATIONAL_CONTEXT", FACTUAL]),
}
flawed_hits = list(flawed_store)

m1 = admit("m1", "Q3 revenue was approximately $46 million.",
            "MODEL_OUTPUT", "UNVERIFIED")
m2 = admit("m2", "Q2 revenue was $43.8M.", "EXTERNAL_SOURCE", "VERIFIED",
            evidence_id="filing_q2", source_family="regulatory_filing")
m3 = admit("m3", "Summary of Q2 verified revenue and Q3 rumor.",
            "MEMORY_DERIVED", "UNVERIFIED", parents=[m1, m2])
governed_store = {m.mid: m for m in (m1, m2, m3)}
governed_hits = factual_recall(governed_store)

print("FLAWED_STORE")
print("m1 origin=MODEL_OUTPUT verification=UNVERIFIED lifecycle=ACTIVE")
print("flawed factual retrieval:", flawed_hits)
print("GOVERNED_STORE")
for m in (m1, m2, m3):
    print(f"{m.mid} origin={m.origin} verification={m.verification} "
          f"lifecycle={m.lifecycle} uses={m.allowed_uses}")
print("factual-evidence retrieval:", governed_hits)


In [ ]:
assert m1.lifecycle == "QUARANTINED"
assert FACTUAL not in m1.allowed_uses
assert m2.lifecycle == "ACTIVE" and FACTUAL in m2.allowed_uses
assert m3.lifecycle == "QUARANTINED" and FACTUAL not in m3.allowed_uses
assert flawed_hits == ["m1"]
assert governed_hits == ["m2"]

print("admission: rumor quarantined, verified source admitted, summary not laundered")


## Read path blocks circular support, permits evidence-indexed recall

Derivation history can stay acyclic while justification loops: invented text is stored, retrieved, and cited as support for itself. Support is admissible only when at least one path terminates outside the claim's own lineage. A memory that indexes durable evidence (`filing_q2`) passes; a memory descended from the claim itself does not. Repeated summarization never launders the rumor into factual rights.


In [ ]:
def support_status(claim_candidate, support):
    independent = [mid for mid, roots in support if claim_candidate not in roots]
    if independent:
        return ("ADMISSIBLE", independent)
    return ("BLOCKED_CIRCULAR", [])


def summarize(parent, mid):
    return admit(mid, "summary of: " + parent.content, "MEMORY_DERIVED",
                  parent.verification, parents=[parent])


q3_support = [("m1", {"cand_v1"})]
q2_support = [("m2", {"filing_q2"})]
q3_status, q3_paths = support_status("cand_v1", q3_support)
q2_status, q2_paths = support_status("cand_v9", q2_support)
print("q3 (self-citation):", q3_status, q3_paths)
print("q2 (evidence-indexed):", q2_status, q2_paths,
      "evidence_id=", m2.evidence_id)

s1 = summarize(m1, "s1")
s2 = summarize(s1, "s2")
s3 = summarize(s2, "s3")
print("laundering chain lifecycles:", [m.lifecycle for m in (m1, s1, s2, s3)])
print("s3 uses:", s3.uses if hasattr(s3, "uses") else s3.allowed_uses)


In [ ]:
assert (q3_status, q3_paths) == ("BLOCKED_CIRCULAR", [])
assert (q2_status, q2_paths) == ("ADMISSIBLE", ["m2"])
assert m2.evidence_id == "filing_q2"
assert all(m.lifecycle == "QUARANTINED" for m in (s1, s2, s3))
assert all(FACTUAL not in m.allowed_uses for m in (s1, s2, s3))
assert s3.parents[0].mid == "s2" and s2.parents[0].mid == "s1"

print("read path: circular self-support blocked, indexed recall permitted")


## Events are not propositions; revocation revalidates instead of deleting blindly

`User said manager is Alice` is a verified observation of an event; `manager(user) = Alice` as a world proposition stays unverified with different capabilities. Revoking a root marks dependents `NEEDS_REVALIDATION` rather than deleting them: the claim slice still covered by an independent root survives. A memory past its `valid_to` loses current-state authority.


In [ ]:
event = MemoryRecord("e1", "User stated that their manager is Alice.",
                        "USER", "VERIFIED", "ACTIVE",
                        ["PERSONALIZATION", "CONVERSATIONAL_CONTEXT"])
prop = admit("p1", "manager(user) = Alice.", "USER", "UNVERIFIED")
print("event:", event.lifecycle, event.allowed_uses)
print("proposition:", prop.lifecycle, prop.allowed_uses)

m1.lifecycle = "REVOKED"
dependents = [d for d in (m3,) if any(p.mid == "m1" for p in d.parents)]
for d in dependents:
    d.lifecycle = "NEEDS_REVALIDATION"
print("after revoke(m1): m3 =", m3.lifecycle)
revalidated = {
    "q2_part": "ACTIVE" if FACTUAL in m2.allowed_uses else "QUARANTINED",
    "q3_part": "QUARANTINED",
}
print("revalidated against remaining roots:", revalidated)

ceo = MemoryRecord("ceo1", "CEO = Bob.", "EXTERNAL_SOURCE", "VERIFIED",
                      "ACTIVE", ["CONVERSATIONAL_CONTEXT", FACTUAL],
                      evidence_id="board_minutes", valid_to="2026-06-30")
stale_authoritative = not (ceo.valid_to is not None and "2026-08-30" > ceo.valid_to)
print("ceo authoritative as of 2026-08-30:", stale_authoritative)


In [ ]:
assert event.lifecycle == "ACTIVE"
assert FACTUAL not in event.allowed_uses
assert prop.lifecycle == "QUARANTINED" and FACTUAL not in prop.allowed_uses
assert m3.lifecycle == "NEEDS_REVALIDATION"
assert revalidated == {"q2_part": "ACTIVE", "q3_part": "QUARANTINED"}
assert stale_authoritative is False
assert factual_recall({"ceo1": ceo}, as_of="2026-08-30") == []

print("event/proposition split; revocation revalidates; stale memory excluded")


## What we earned

Stored is not verified and retrieved is not admissible: the admission gate quarantines unverified model output, derivation preserves unresolved provenance instead of laundering it, the read path blocks circular self-citation while permitting evidence-indexed recall, and revocation revalidates dependents against surviving roots.

Notebook 15 / Chapter 15 assembles these gates with policy, recovery, and enforcement into one system that distrusts its model by design.
